# Olist Brazilian E-Commerce — Exploratory Analysis
**Team 3** · Clement, Chih Chang, HengQing, BenJ, Danny

This notebook runs against the **marts** produced by the dbt layer
(`analytics_marts.fct_orders`, `fct_category_performance`, `fct_monthly_revenue`,
`dim_customers`). It assumes the pipeline has already run
(`meltano run` → `dbt build`) and the DuckDB warehouse exists at `../data/olist.duckdb`.

**Business questions**
1. How big is the business and how is it growing?
2. What drives (and destroys) customer satisfaction?
3. Where is revenue concentrated — by category and geography?
4. How strong is customer retention?

In [ ]:
import duckdb, pandas as pd
import matplotlib.pyplot as plt
plt.rcParams.update({'figure.dpi':110,'font.size':11,'axes.spines.top':False,'axes.spines.right':False})
con = duckdb.connect('../data/olist.duckdb')
con.execute("SELECT count(*) FROM analytics_marts.fct_orders").fetchone()

## 1. Business size & growth

In [ ]:
kpis = con.execute('''
SELECT count(*) orders, round(sum(gmv),0) gmv,
       round(sum(gmv)/count(distinct order_id),2) aov,
       round(avg(review_score),2) avg_review
FROM analytics_marts.fct_orders''').df()
kpis

In [ ]:
m = con.execute('''SELECT order_month, gmv, orders FROM analytics_marts.fct_monthly_revenue
WHERE order_month BETWEEN '2017-01' AND '2018-08' ORDER BY 1''').df()
ax = m.plot(x='order_month', y='gmv', marker='o', figsize=(10,4), legend=False, color='#1f3b57')
ax.set_title('Monthly GMV'); ax.set_ylabel('R$'); plt.xticks(rotation=45); plt.tight_layout()

**Finding:** GMV grew from ~R$120K (Jan 2017) to ~R$1M/month by mid-2018 —
roughly **8x in 18 months**, with a clear Black Friday spike in Nov 2017.

## 2. What drives satisfaction? Delivery is the lever.

In [ ]:
late = con.execute('''SELECT is_late, round(avg(review_score),2) avg_score, count(*) n
FROM analytics_marts.fct_orders WHERE delivered_at IS NOT NULL GROUP BY 1 ORDER BY 1''').df()
late

In [ ]:
ax = late.assign(label=late.is_late.map({False:'On-time',True:'Late'})).plot(
    x='label', y='avg_score', kind='bar', legend=False, color=['#2a9d8f','#e76f51'], figsize=(6,4))
ax.set_title('Avg review score: on-time vs late'); ax.set_ylim(0,5); plt.xticks(rotation=0); plt.tight_layout()

**Finding:** On-time orders average **4.29**, late orders **2.57** — a **1.7-point** drop.
**8.1%** of orders arrive late, and overall avg delivery time is **12.5 days**.
Delivery reliability is the single biggest controllable driver of satisfaction.

## 3. Where is revenue concentrated?

In [ ]:
cat = con.execute('''SELECT category, revenue, avg_review_score
FROM analytics_marts.fct_category_performance ORDER BY revenue DESC LIMIT 10''').df()
cat

In [ ]:
geo = con.execute('''SELECT customer_state, round(sum(gmv),0) gmv,
round(100.0*sum(gmv)/sum(sum(gmv)) over(),1) pct
FROM analytics_marts.fct_orders GROUP BY 1 ORDER BY 2 DESC LIMIT 8''').df()
geo

**Finding:** The top category (health & beauty) and São Paulo state dominate.
**SP alone is ~38% of GMV** — a concentration risk and an expansion opportunity.

## 4. Customer retention

In [ ]:
ret = con.execute('''SELECT round(100.0*sum(CASE WHEN is_repeat THEN 1 ELSE 0 END)/count(*),2) repeat_pct
FROM analytics_marts.dim_customers''').df()
ret

**Finding:** Only **~3.1%** of customers place a second order. Acquisition is working;
**retention is the biggest untapped growth lever** — even a small lift compounds on an 8x-growth base.

## Summary of recommendations
1. **Fix delivery reliability** in the worst lanes — biggest satisfaction ROI.
2. **Launch a retention program** (post-purchase nudges, loyalty) — 3.1% repeat is very low.
3. **Geographic expansion** beyond SP, paired with regional fulfillment.
4. **Double down on top categories** while monitoring review scores per category.

In [ ]:
con.close()